In [1]:
import numpy as np
import sys

# NumPyがどこから読み込まれているか、パスを直接出力
print(f"NumPy Version: {np.__version__}")
print(f"NumPy Path: {np.__file__}")

# もしここで 1.26より上が出たら、Jupyterのカーネル接続先が狂っています
import numba
print("Numba loaded successfully!")

NumPy Version: 1.24.4
NumPy Path: /home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/site-packages/numpy/__init__.py
Numba loaded successfully!


In [ ]:
import pandas as pd
import numpy as np
import shap
import os
import random
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

# --- [1] パス・シード設定 ---
# ターゲット: Jsc / ファイル: m_all_OH_rebuilt.csv / アルゴリズム: GPR_Linear
file_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/m_all_OH_rebuilt.csv"
out_dir = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/GPR_Linear_SHAP_Jsc_m_all"
os.makedirs(out_dir, exist_ok=True)

# 再現性のための完全シード固定
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# --- [2] データ読み込みと前処理 ---
print(f"Loading data: {os.path.basename(file_path)}...")
# index_col=0 により1列目のIDを特徴量から確実に除外
df = pd.read_csv(file_path, index_col=0)
df_num = df.select_dtypes(include=[np.number]).dropna()

target = "Jsc"

# 除外設定の厳格な定義 (博士論文基準)
TARGET_CANDIDATES = ["PCE", "PCEmax", "PCEavg", "Jsc", "Voc", "FF"]
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

# 1. ターゲット候補、不適切変数、および ID由来の "X" 列を確実に除外
features = [c for c in df_num.columns if c not in TARGET_CANDIDATES and c not in INAPPROPRIATE_VARS and c != "X"]

# 2. 物理データパターンの除外
features = [f for f in features if not any(pat in f for pat in PHYSICAL_EXCLUDE_PATTERNS)]

# 3. SD（標準偏差）が 0 の列を完全に排除（計算の安定化・行列の特異性回避）
features = [f for f in features if df_num[f].std() > 0]

print(f"Final Features count for {target} (m_all): {len(features)}")

X_raw = df_num[features]
y = df_num[target]

# スケーリング（-1 to 1）
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features, index=X_raw.index)

# --- [3] GPR (Linear) 学習 ---
print(f"Training GPR (Linear Kernel) for {target}...")
# Linear Kernel (DotProduct) + WhiteKernel (ノイズ吸収用)
kernel = DotProduct() + WhiteKernel()
model = GaussianProcessRegressor(kernel=kernel, random_state=SEED)
model.fit(X_scaled, y)

# モデル精度評価 (訓練データに対するスコア)
y_pred = model.predict(X_scaled)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("-" * 35)
print(f"Model Performance ({target}):")
print(f"  R2 Score: {r2:.4f}")
print(f"  RMSE:     {rmse:.4f}")
print("-" * 35)

# --- [4] SHAP計算 (KernelExplainer) ---
print(f"Calculating SHAP values for {target} (KernelExplainer)...")
# 背景データのサンプリングを固定 (random_state=SEED)
X_summary = shap.sample(X_scaled, 10, random_state=SEED)
explainer = shap.KernelExplainer(model.predict, X_summary)

# l1_reg=False により、数値の不安定化を防止
shap_values = explainer.shap_values(X_scaled, l1_reg=False)

# --- [5] 保存と可視化 ---
# 1. 重要度CSVの保存
importance_df = pd.DataFrame({
    "Feature": features,
    "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0)
}).sort_values("SHAP_MeanAbs", ascending=False)

out_csv = os.path.join(out_dir, f"{target}_m_all_OH_GPR_Linear_SHAP.csv")
importance_df.to_csv(out_csv, index=False)

# 2. 精度サマリーの保存
with open(os.path.join(out_dir, f"{target}_Performance_Summary_GPR.txt"), "w") as f:
    f.write(f"Target: {target}\nFile: {os.path.basename(file_path)}\n")
    f.write(f"Algorithm: GPR (Linear Kernel)\n")
    f.write(f"R2 Score: {r2:.4f}\nRMSE: {rmse:.4f}\n")
    f.write(f"Features used: {len(features)}\n")

# 3. SHAP Beeswarm Plot の保存
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_scaled, show=False)
plt.title(f"SHAP Summary: {target} (GPR Linear - R2={r2:.3f})")
plt.savefig(os.path.join(out_dir, f"SHAP_Beeswarm_{target}_m_all_OH_GPR.png"), bbox_inches='tight', dpi=300)
plt.close()

print(f"Success! All results for {target} (GPR) saved in: {out_dir}")

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Loading data: m_all_OH_rebuilt.csv...
Final Features count for Jsc (m_all): 430
Training GPR (Linear Kernel) for Jsc...
-----------------------------------
Model Performance (Jsc):
  R2 Score: 0.8771
  RMSE:     1.6618
-----------------------------------
Calculating SHAP values for Jsc (KernelExplainer)...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏| 1040/1045 [11:25<00:02,  1.93it/s]